In [30]:
import collections as col
import pathlib as pl

import pandas as pd

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

glob_pattern = "**/*refOriented.haplotype*.fasta.gz"

search_path = CONFIG["local_hilbert_prefix"].joinpath(CONFIG["hilbert_verkko_assembly_folder"])
assert search_path.is_dir(), f"Search path unavailable: {search_path}"

sample_annotation = CONFIG["project_repo"].joinpath(
    "annotation", "norm", "samples_1kg_pedigree.ext.tsv"
).resolve(strict=True)

haplogroup_annotation = CONFIG["project_repo"].joinpath(
    "annotation", "raw", "hallast_2023_nature.supp-table-1.sample-desc.tsv"
).resolve(strict=True)

haplogroups = []
with open(haplogroup_annotation, "r") as listing:
    for line in listing:
        columns = line.strip().split()
        sample = columns[1]
        haplogroup = columns[2]
        if sample == "T2T-Y":
            sample = "NA24385"
            haplogroup = columns[3]
        hg_root = haplogroup[:2]
        haplogroups.append((sample, haplogroup.strip(), hg_root))
haplogroups = pd.DataFrame.from_records(haplogroups, columns=["sample", "haplogroup", "hg_short"])
haplogroups.set_index("sample", inplace=True)

samples = pd.read_csv(sample_annotation, sep="\t", header=0, comment="#")
samples.set_index("individual_id", inplace=True)

asm_norm = {
    "verkko-hi-c": "vrk-hic",
    "verkko-thic": "vrk-thc"
}

au_norm = {
    "haplotype1": "hap1",
    "haplotype2": "hap2"
}

norm_sample = {
    "HG002": "NA24385",
    "HG005": "NA24631"
}

rows = col.defaultdict(dict)
for fasta_file in search_path.glob(glob_pattern):
    sample = fasta_file.name.split(".")[0]
    sample = norm_sample.get(sample, sample)
    assembly_type = asm_norm[fasta_file.parent.parent.name]
    asm_unit = au_norm[fasta_file.name.split(".")[3]]
    try:
        sample_sex = samples.at[sample, "karyotype"]
    except KeyError:
        print("Missing: ", sample)
        continue
    rows[sample]["sample_sex"] = sample_sex
    rows[sample][f"asm_{asm_unit}"] = replace_path_prefix(fasta_file)
    rows[sample]["assembly_type"] = assembly_type
    

assemblies = pd.DataFrame.from_dict(rows, orient="index")
assemblies.index.name = "sample"
assemblies.sort_index(inplace=True)
assemblies = assemblies[["sample_sex", "assembly_type", "asm_hap1", "asm_hap2"]]

assemblies = assemblies.join(haplogroups)

select_females = assemblies["sample_sex"] == "female"
select_males = assemblies["sample_sex"] == "male"
assemblies.loc[select_females, "haplogroup"] = "XX"
assemblies.loc[select_females, "hg_short"] = "XX"

assemblies["study_generation"] = 2025

select_known_males = ~pd.isnull(assemblies["haplogroup"])
assemblies.loc[select_known_males & select_males, "study_generation"] = 2023
assemblies["haplogroup"] = assemblies["haplogroup"].fillna("unknown", inplace=False)
assemblies["hg_short"] = assemblies["hg_short"].fillna("UN", inplace=False)

assemblies = assemblies[
    [
        "sample_sex", "assembly_type", "hg_short", "haplogroup",
        "study_generation", "asm_hap1", "asm_hap2"
    ]
]

sample_sheet_file = CONFIG["project_repo"].joinpath("samples", "verkko_assemblies.tsv")
sample_sheet_file.parent.mkdir(exist_ok=True, parents=True)

with open(sample_sheet_file, "w") as table:
    _ = table.write(f"# {TIMESTAMP}\n")
    _ = table.write(f"# N={assemblies.shape[0]}\n")
    assemblies.to_csv(table, sep="\t", header=True, index=True)
    